# 04 — 4-bit QLoRA Fine-Tuning on Colab Free T4 (`Qwen/Qwen3-4B-Instruct-2507`)
# Huấn luyện 4-bit QLoRA trên Google Colab T4 Miễn phí

**English:**
Fine-tunes `Qwen/Qwen3-4B-Instruct-2507` with 4-bit NF4 QLoRA (`fp16` compute, `paged_adamw_8bit`, `gradient_checkpointing=True`), saves checkpoints every 10 steps (`save_total_limit=3`) directly to Google Drive (`/content/drive/MyDrive/mimamori_companion_ai/checkpoints`), supports `resume_from_checkpoint=True` for disconnect recovery, logs train/validation loss, saves the final LoRA adapter + tokenizer + `training_metadata.json`, and runs post-training test-split and benchmark evaluation.

**Tiếng Việt:**
Huấn luyện tinh chỉnh (QLoRA 4-bit NF4, `fp16`, `paged_adamw_8bit`) mô hình `Qwen/Qwen3-4B-Instruct-2507` cho vai trò bạn đồng hành người cao tuổi Việt Nam. Checkpoint được lưu định kỳ trực tiếp lên Google Drive để khôi phục nếu Colab ngắt kết nối (`resume_from_checkpoint=True`), sau đó lưu LoRA adapter + tokenizer + `training_metadata.json` và đánh giá trên tập test/benchmark.

---
> ## PRE-FLIGHT SAFETY & REVIEW GATE / CỔNG KIỂM TRA TRƯỚC KHI HUẤN LUYỆN
> **STOP / DỪNG LẠI:**
> Do **NOT** run full production fine-tuning until the synthetic seed dataset (`synthetic_companion_seed.jsonl`, currently `reviewed=false`, 16 train / 8 val examples) has been **reviewed by clinical/geriatric reviewers, expanded (>= 200 examples), and marked `reviewed=true`**!
> - By default, both `ALLOW_SYNTHETIC_SMOKE_TEST` and `CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING` are **`False`**, blocking accidental training on unreviewed synthetic data.
> - Only flip `ALLOW_SYNTHETIC_SMOKE_TEST = True` for a bounded pipeline smoke test on Colab T4, or `CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING = True` after replacing with reviewed dialogues.


## Step 1: Mount Drive, Load Configs & Check T4 15GB VRAM Safeguards
## Bước 1: Kết nối Drive, Đọc Cấu hình & Kiểm tra Bảo vệ VRAM T4 15GB


In [ ]:
import os
import sys
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT_DEFAULT = Path("/content/drive/MyDrive/mimamori_companion_ai")

# Optional: set AI_TRAINING_DIR explicitly if your repo lives in a custom folder
CUSTOM_AI_TRAINING_DIR = os.environ.get("MIMAMORI_AI_TRAINING_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("[Drive] Google Drive mounted at /content/drive")
else:
    print("[Local] Not running inside Google Colab; Drive mount skipped.")


def resolve_ai_training_dir() -> Path:
    candidates: list[Path] = []
    if CUSTOM_AI_TRAINING_DIR:
        candidates.append(Path(CUSTOM_AI_TRAINING_DIR))
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / "ai-training",
        cwd.parent / "ai-training",
        Path("/content/mimamori/ai-training"),
        Path("/content/ai-training"),
        Path("/content/drive/MyDrive/mimamori/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/repo/ai-training"),
        Path("/content/drive/MyDrive/mimamori_companion_ai/ai-training"),
    ])
    for cand in candidates:
        if (cand / "configs" / "model_config.yaml").exists() and (cand / "configs" / "training_config.yaml").exists():
            return cand.resolve()
    raise FileNotFoundError(
        "Could not locate 'ai-training/configs/model_config.yaml'. "
        "In Colab, clone/upload the repository to /content/mimamori or Google Drive, "
        "or set CUSTOM_AI_TRAINING_DIR / MIMAMORI_AI_TRAINING_DIR."
    )


AI_TRAINING_DIR = resolve_ai_training_dir()
if str(AI_TRAINING_DIR / "scripts") not in sys.path:
    sys.path.insert(0, str(AI_TRAINING_DIR / "scripts"))

import yaml

MODEL_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "model_config.yaml"
TRAINING_CONFIG_PATH = AI_TRAINING_DIR / "configs" / "training_config.yaml"
with MODEL_CONFIG_PATH.open("r", encoding="utf-8") as f:
    MODEL_CFG = yaml.safe_load(f)
with TRAINING_CONFIG_PATH.open("r", encoding="utf-8") as f:
    TRAIN_CFG = yaml.safe_load(f)

PATHS_CFG = TRAIN_CFG["paths"]
if IN_COLAB:
    DRIVE_ROOT = Path(PATHS_CFG["drive_root"])
    DRIVE_CHECKPOINTS_DIR = Path(PATHS_CFG["drive_checkpoints_dir"])
    DRIVE_FINAL_ADAPTER_DIR = Path(PATHS_CFG["drive_final_adapter_dir"])
    DRIVE_EVAL_OUTPUTS_DIR = Path(PATHS_CFG["drive_eval_outputs_dir"])
    DRIVE_PROCESSED_DATA_DIR = Path(PATHS_CFG["drive_processed_data_dir"])
else:
    local_drive_sim = AI_TRAINING_DIR / "outputs" / "_local_drive_mirror"
    DRIVE_ROOT = local_drive_sim
    DRIVE_CHECKPOINTS_DIR = local_drive_sim / "checkpoints"
    DRIVE_FINAL_ADAPTER_DIR = local_drive_sim / "adapters" / "qwen3-4b-companion-lora"
    DRIVE_EVAL_OUTPUTS_DIR = AI_TRAINING_DIR / PATHS_CFG["local_outputs_dir"]
    DRIVE_PROCESSED_DATA_DIR = AI_TRAINING_DIR / "datasets" / "processed"

for d in (DRIVE_ROOT, DRIVE_CHECKPOINTS_DIR, DRIVE_FINAL_ADAPTER_DIR, DRIVE_EVAL_OUTPUTS_DIR, DRIVE_PROCESSED_DATA_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"[Repo] AI_TRAINING_DIR      = {AI_TRAINING_DIR}")
print(f"[Drive] DRIVE_ROOT          = {DRIVE_ROOT}")
print(f"[Drive] DRIVE_EVAL_OUTPUTS  = {DRIVE_EVAL_OUTPUTS_DIR}")
print(f"[Drive] DRIVE_PROCESSED     = {DRIVE_PROCESSED_DATA_DIR}")


def inspect_gpu_and_t4_safeguards(train_cfg: dict) -> dict:
    """Detect GPU/VRAM and return Free Colab T4 15GB safe hyperparameters."""
    t4_target = train_cfg["hardware_target"]
    base_train = dict(train_cfg["training"])
    fallback = dict(train_cfg["t4_oom_fallback"])
    info = {
        "cuda_available": False,
        "gpu_name": "CPU",
        "vram_gb": 0.0,
        "compute_capability": None,
        "supports_bf16": False,
        "supports_flash_attention_2": False,
        "use_t4_oom_fallback": False,
        "effective_fp16": True,
        "effective_bf16": False,
        "effective_batch_size": fallback["per_device_train_batch_size"],
        "effective_grad_accum": fallback["gradient_accumulation_steps"],
        "effective_max_seq_length": fallback["max_seq_length"],
        "warnings": [],
    }

    try:
        import torch
    except ImportError:
        info["warnings"].append("PyTorch is not installed; GPU checks unavailable.")
        return info

    if not torch.cuda.is_available():
        info["warnings"].append(
            "No CUDA GPU detected. In Colab, select Runtime -> Change runtime type -> T4 GPU (Free tier; NEVER enable paid GPU/billing)."
        )
        return info

    props = torch.cuda.get_device_properties(0)
    vram_gb = round(props.total_memory / (1024 ** 3), 2)
    major, minor = torch.cuda.get_device_capability(0)
    supports_ampere = (major, minor) >= (8, 0)

    info["cuda_available"] = True
    info["gpu_name"] = props.name
    info["vram_gb"] = vram_gb
    info["compute_capability"] = f"{major}.{minor}"
    # Enforce T4 safe precision policy per hardware_target (fp16=True, bf16=False on Colab T4)
    info["supports_bf16"] = bool(t4_target.get("supports_bf16", False)) and supports_ampere
    info["supports_flash_attention_2"] = bool(t4_target.get("supports_flash_attention_2", False)) and supports_ampere

    if vram_gb <= 15.5:
        info["warnings"].append(
            f"Detected <=15.5GB GPU ({props.name}, {vram_gb} GB, sm_{major}{minor}). "
            "Applying Free T4 safeguards: fp16=True, bf16=False, gradient_checkpointing=True, paged_adamw_8bit."
        )
    if vram_gb < 14.5:
        info["use_t4_oom_fallback"] = True
        info["effective_batch_size"] = fallback["batch_size"]
        info["effective_grad_accum"] = fallback["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = fallback["max_seq_length"]
        info["warnings"].append(
            f"VRAM ({vram_gb} GB) < 14.5 GB -> auto-switching to t4_oom_fallback: "
            f"batch_size={fallback['batch_size']}, grad_accum={fallback['gradient_accumulation_steps']}, "
            f"max_seq_length={fallback['max_seq_length']}."
        )
    else:
        info["effective_batch_size"] = base_train["per_device_train_batch_size"]
        info["effective_grad_accum"] = base_train["gradient_accumulation_steps"]
        info["effective_max_seq_length"] = base_train["max_seq_length"]

    if not supports_ampere:
        info["effective_fp16"] = True
        info["effective_bf16"] = False
    return info


GPU_SAFEGUARD = inspect_gpu_and_t4_safeguards(TRAIN_CFG)
for _w in GPU_SAFEGUARD["warnings"]:
    print(f"[T4 Safeguard] {_w}")
print("[Hardware Profile]", GPU_SAFEGUARD)


## Step 2: Mandatory Pre-Flight Dataset Review Gate & Library Compatibility Check
## Bước 2: Cổng Kiểm tra Duyệt Dữ liệu & Tương thích Thư viện Bắt buộc

Enforces `safety_and_review_gate` from `configs/training_config.yaml` and verifies `transformers >= 4.51.0`, `peft`, `trl`, `bitsandbytes`, `accelerate`, `datasets`, plus CUDA availability.


In [ ]:
import importlib
import json
import re
from validate_dataset import load_jsonl

# --- USER SAFETY GATE FLAGS (KEEP False UNTIL INTENTIONAL COLAB RUN) ---
ALLOW_SYNTHETIC_SMOKE_TEST = False
CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING = False
FORCE_T4_SAFE_PROFILE = False  # Set True to force batch_size=1, grad_accum=16, max_seq_length=768 immediately

# Locate processed splits (prefer local repo, fallback to Drive backup)
def _locate_split(filename: str) -> Path:
    local_p = AI_TRAINING_DIR / "datasets" / "processed" / filename
    drive_p = DRIVE_PROCESSED_DATA_DIR / filename
    if local_p.exists():
        return local_p
    if drive_p.exists():
        return drive_p
    raise FileNotFoundError(f"Missing processed split '{filename}'. Run 02_dataset_preparation.ipynb first.")


TRAIN_PATH = _locate_split("train.jsonl")
VAL_PATH = _locate_split("val.jsonl")
TEST_PATH = _locate_split("test.jsonl")
EVAL_PATH = AI_TRAINING_DIR / PATHS_CFG["local_eval_benchmark_file"]
if not EVAL_PATH.exists():
    EVAL_PATH = DRIVE_PROCESSED_DATA_DIR / "eval_benchmark.jsonl"

train_records, _ = load_jsonl(TRAIN_PATH)
val_records, _ = load_jsonl(VAL_PATH)
test_records, _ = load_jsonl(TEST_PATH)

reviewed_train = sum(1 for r in train_records if r.get("reviewed") is True)
unreviewed_train = len(train_records) - reviewed_train
gate_cfg = TRAIN_CFG["safety_and_review_gate"]
min_rec_examples = int(gate_cfg["min_recommended_reviewed_examples"])

print(f"Dataset splits: train={len(train_records)}, val={len(val_records)}, test={len(test_records)}")
print(f"Train reviewed=true: {reviewed_train}/{len(train_records)} | reviewed=false: {unreviewed_train}")
print(f"Recommended minimum reviewed training examples for production: {min_rec_examples}")

if unreviewed_train > 0 or len(train_records) < min_rec_examples:
    print(
        "\n[PRE-FLIGHT WARNING] Dataset is still synthetic/unreviewed or below recommended size! "
        "Do NOT promote resulting weights to production ai-service/."
    )
    if not ALLOW_SYNTHETIC_SMOKE_TEST and not CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING:
        raise RuntimeError(
            "Pre-flight gate halted training intentionally: dataset contains reviewed=false synthetic examples. "
            "To run a bounded Colab T4 pipeline smoke test, explicitly set ALLOW_SYNTHETIC_SMOKE_TEST = True. "
            "For full training, replace datasets with reviewed=true dialogues first."
        )
elif not CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING and not ALLOW_SYNTHETIC_SMOKE_TEST:
    raise RuntimeError(
        "Set CONFIRM_REVIEWED_DATASET_FOR_FULL_TRAINING = True to start training."
    )

# Check CUDA and required training dependencies
if not GPU_SAFEGUARD["cuda_available"]:
    raise RuntimeError(
        "CUDA GPU is required for 4-bit QLoRA training. Switch Colab runtime to Free T4 GPU (never paid billing)."
    )

def _ver_ge(cur: str, req: str) -> bool:
    m1 = re.match(r"^\s*(\d+)\.(\d+)(?:\.(\d+))?", str(cur))
    m2 = re.match(r"^\s*(\d+)\.(\d+)(?:\.(\d+))?", str(req))
    t1 = tuple(int(x or 0) for x in m1.groups()) if m1 else (0, 0, 0)
    t2 = tuple(int(x or 0) for x in m2.groups()) if m2 else (0, 0, 0)
    return t1 >= t2

lib_versions = {}
for pkg, min_v in [("transformers", MODEL_CFG["primary_model"]["min_transformers_version"]), ("peft", "0.14.0"), ("trl", "0.16.0"), ("bitsandbytes", "0.45.0"), ("accelerate", "1.5.0"), ("datasets", "3.2.0")]:
    mod = importlib.import_module(pkg)
    ver = getattr(mod, "__version__", "0.0.0")
    lib_versions[pkg] = ver
    if not _ver_ge(ver, min_v):
        raise RuntimeError(f"{pkg}=={ver} is below required >={min_v}")

IS_SYNTHETIC_SMOKE_TEST = bool(unreviewed_train > 0 or len(train_records) < min_rec_examples)
print("[Pre-Flight Gate Passed]", {"IS_SYNTHETIC_SMOKE_TEST": IS_SYNTHETIC_SMOKE_TEST, "libs": lib_versions})


## Architecture Compatibility Notice: Why `Qwen/Qwen3-4B-Instruct-2507` (and NOT `Qwen/Qwen3.5-4B` QLoRA on T4)
## Lưu ý Tương thích Kiến trúc: Rủi ro khi QLoRA 4-bit với `Qwen/Qwen3.5-4B`

> **CRITICAL COMPATIBILITY WARNING / CẢNH BÁO TƯƠNG THÍCH QUAN TRỌNG:**
> - **Primary Model (`Qwen/Qwen3-4B-Instruct-2507`)**: Pure dense Transformer (`Qwen3ForCausalLM`, 36 layers, non-thinking mode). **4-bit NF4 QLoRA** (`fp16` compute, `paged_adamw_8bit`) fits safely within **~6.5–9.5 GB VRAM** on a Free 15GB T4 using standard target modules `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]`.
> - **Why `Qwen/Qwen3.5-4B` is NOT trained with 4-bit QLoRA here**:
>   - `Qwen/Qwen3.5-4B` is an early-fusion VLM + hybrid 3:1 `GatedDeltaNet` `linear_attn` (24 layers) + `full_attention` (8 layers) + vision (`24` blocks), requiring `transformers >= 5.2.0` and `enable_thinking=False`.
>   - **4-bit QLoRA divergence risk**: `GatedDeltaNet` `linear_attn` + `conv1d` suffers severe quantization divergence and `bitsandbytes` split-projection issues under 4-bit QLoRA (`qlora_4bit_recommended: false` in `configs/model_config.yaml`).
>   - Pure `fp16` LoRA also risks `NaN` losses (`mamba_ssm_dtype=float32`), requires extra `linear_attn` target modules (`in_proj_qkv, in_proj_z, in_proj_a, in_proj_b, out_proj`) while excluding `conv1d` & `model.visual.*`, and runs slowly without `causal-conv1d` / `flash-linear-attention` kernels (~10–12.5+ GB VRAM, tight on T4 15GB).


In [ ]:
comp_cfg = MODEL_CFG["comparison_model"]
print("[Qwen3.5-4B Advisory Summary from model_config.yaml]")
print("  model_id               :", comp_cfg["id"])
print("  min_transformers       :", comp_cfg["min_transformers_version"])
print("  qlora_4bit_recommended :", comp_cfg["qlora_4bit_recommended"])
print("  linear_attn_modules    :", comp_cfg["lora_target_modules_linear_attn"])
print("  excluded_modules       :", comp_cfg["excluded_modules"])
print("  notes                  :", comp_cfg["instability_risk_notes"].strip())


## Step 3: Load `Qwen/Qwen3-4B-Instruct-2507` (4-bit NF4) & Format Chat Splits
## Bước 3: Tải `Qwen/Qwen3-4B-Instruct-2507` (4-bit NF4) & Định dạng Dữ liệu Hội thoại


In [ ]:
import torch
from datasets import Dataset
from peft import LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL_ID = TRAIN_CFG["experiment"]["base_model_id"]
QUANT_CFG = TRAIN_CFG["quantization"]
LORA_CFG = TRAIN_CFG["lora"]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=bool(QUANT_CFG["load_in_4bit"]),
    bnb_4bit_quant_type=str(QUANT_CFG["bnb_4bit_quant_type"]),
    bnb_4bit_compute_dtype=torch.float16,  # T4 sm_75 requires float16 (no native bf16)
    bnb_4bit_use_double_quant=bool(QUANT_CFG["bnb_4bit_use_double_quant"]),
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=bool(TRAIN_CFG["training"]["gradient_checkpointing"]))

peft_config = LoraConfig(
    r=int(LORA_CFG["r"]),
    lora_alpha=int(LORA_CFG["lora_alpha"]),
    lora_dropout=float(LORA_CFG["lora_dropout"]),
    bias=str(LORA_CFG["bias"]),
    task_type=str(LORA_CFG["task_type"]),
    target_modules=list(LORA_CFG["target_modules"]),
)


def to_chatml_dataset(records: list[dict]) -> Dataset:
    rendered_rows = []
    for rec in records:
        text = tokenizer.apply_chat_template(
            rec["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        rendered_rows.append({"id": rec["id"], "category": rec["category"], "text": text})
    return Dataset.from_list(rendered_rows)


train_dataset = to_chatml_dataset(train_records)
val_dataset = to_chatml_dataset(val_records)
test_dataset = to_chatml_dataset(test_records)

print(f"Prepared HF Datasets -> train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")
print(f"LoRA target_modules  -> {peft_config.target_modules}")


## Step 4: Configure Drive Checkpoints, Auto-Resume & OOM-Safe Training Loop
## Bước 4: Cấu hình Lưu Checkpoint lên Drive, Tự động Khôi phục & Vòng lặp Chống Tràn VRAM (OOM)

- Automatically checks `DRIVE_CHECKPOINTS_DIR` (`/content/drive/MyDrive/mimamori_companion_ai/checkpoints`) for `checkpoint-*` folders to resume after any Colab disconnect (`resume_from_checkpoint=True`).
- If `torch.cuda.OutOfMemoryError` or CUDA OOM occurs during standard T4 training (`batch_size=2, grad_accum=8, max_seq_length=1024`), automatically clears CUDA cache and retries with `t4_oom_fallback` (`batch_size=1, grad_accum=16, max_seq_length=768, fp16=True, bf16=False`).


In [ ]:
import gc
import inspect
import math
from transformers.trainer_utils import get_last_checkpoint
from trl import SFTConfig, SFTTrainer


def build_sft_trainer(profile_name: str, batch_size: int, eval_batch_size: int, grad_accum: int, max_seq_len: int):
    t_cfg = TRAIN_CFG["training"]
    sft_sig = inspect.signature(SFTConfig.__init__).parameters
    trainer_sig = inspect.signature(SFTTrainer.__init__).parameters

    epochs = int(t_cfg["num_train_epochs"])
    steps_per_epoch = max(1, math.ceil(len(train_dataset) / max(1, int(batch_size) * int(grad_accum))))
    est_total_steps = max(1, steps_per_epoch * epochs)
    cfg_eval_steps = int(t_cfg["eval_steps"])
    cfg_save_steps = int(t_cfg["save_steps"])
    cfg_log_steps = int(t_cfg["logging_steps"])

    # Ensure eval & checkpointing trigger at least once on small smoke datasets so load_best_model_at_end succeeds
    eff_eval_steps = cfg_eval_steps if est_total_steps >= cfg_eval_steps else max(1, est_total_steps)
    eff_save_steps = cfg_save_steps if (est_total_steps >= cfg_save_steps and cfg_save_steps % eff_eval_steps == 0) else eff_eval_steps
    eff_log_steps = cfg_log_steps if est_total_steps >= cfg_log_steps else 1

    cfg_kwargs = {
        "output_dir": str(DRIVE_CHECKPOINTS_DIR),
        "per_device_train_batch_size": int(batch_size),
        "per_device_eval_batch_size": int(eval_batch_size),
        "gradient_accumulation_steps": int(grad_accum),
        "learning_rate": float(t_cfg["learning_rate"]),
        "num_train_epochs": epochs,
        "lr_scheduler_type": str(t_cfg["lr_scheduler_type"]),
        "warmup_ratio": float(t_cfg["warmup_ratio"]),
        "weight_decay": float(t_cfg["weight_decay"]),
        "max_grad_norm": float(t_cfg["max_grad_norm"]),
        "fp16": True,   # Enforce T4 sm_75 fp16=True, bf16=False
        "bf16": False,
        "optim": str(t_cfg["optim"]),
        "gradient_checkpointing": bool(t_cfg["gradient_checkpointing"]),
        "logging_steps": eff_log_steps,
        "eval_steps": eff_eval_steps,
        "save_strategy": str(t_cfg["save_strategy"]),
        "save_steps": eff_save_steps,
        "save_total_limit": int(t_cfg["save_total_limit"]),
        "load_best_model_at_end": bool(t_cfg["load_best_model_at_end"]),
        "metric_for_best_model": str(t_cfg["metric_for_best_model"]),
        "greater_is_better": bool(t_cfg["greater_is_better"]),
        "seed": int(TRAIN_CFG["experiment"]["seed"]),
        "report_to": str(t_cfg["report_to"]),
    }
    if "gradient_checkpointing_kwargs" in sft_sig and "gradient_checkpointing_kwargs" in t_cfg:
        cfg_kwargs["gradient_checkpointing_kwargs"] = dict(t_cfg["gradient_checkpointing_kwargs"])
    if "eval_strategy" in sft_sig:
        cfg_kwargs["eval_strategy"] = str(t_cfg["eval_strategy"])
    elif "evaluation_strategy" in sft_sig:
        cfg_kwargs["evaluation_strategy"] = str(t_cfg["eval_strategy"])
    if "max_length" in sft_sig:
        cfg_kwargs["max_length"] = int(max_seq_len)
    elif "max_seq_length" in sft_sig:
        cfg_kwargs["max_seq_length"] = int(max_seq_len)
    if "packing" in sft_sig:
        cfg_kwargs["packing"] = bool(t_cfg["packing"])
    if "dataset_text_field" in sft_sig:
        cfg_kwargs["dataset_text_field"] = "text"

    sft_config = SFTConfig(**cfg_kwargs)
    trainer_kwargs = {
        "model": model,
        "args": sft_config,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "peft_config": peft_config,
    }
    if "processing_class" in trainer_sig:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_sig:
        trainer_kwargs["tokenizer"] = tokenizer
    if "dataset_text_field" in trainer_sig:
        trainer_kwargs["dataset_text_field"] = "text"
    if "max_seq_length" in trainer_sig:
        trainer_kwargs["max_seq_length"] = int(max_seq_len)

    print(f"[SFTTrainer] Profile={profile_name} | batch={batch_size} | grad_accum={grad_accum} | max_seq={max_seq_len} | fp16=True | bf16=False")
    return SFTTrainer(**trainer_kwargs)


use_safe_first = bool(FORCE_T4_SAFE_PROFILE or GPU_SAFEGUARD["use_t4_oom_fallback"])
t_cfg = TRAIN_CFG["training"]
fb_cfg = TRAIN_CFG["t4_oom_fallback"]

last_ckpt = get_last_checkpoint(str(DRIVE_CHECKPOINTS_DIR)) if TRAIN_CFG["training"].get("resume_from_checkpoint", True) else None
if last_ckpt:
    print(f"[Resume] Found existing Drive checkpoint: {last_ckpt}")
else:
    print("[Fresh Run] No previous checkpoint found in Drive checkpoints dir.")

effective_profile = "t4_oom_fallback" if use_safe_first else "standard_t4"
active_params = (
    (fb_cfg["per_device_train_batch_size"], fb_cfg["per_device_eval_batch_size"], fb_cfg["gradient_accumulation_steps"], fb_cfg["max_seq_length"])
    if use_safe_first
    else (t_cfg["per_device_train_batch_size"], t_cfg["per_device_eval_batch_size"], t_cfg["gradient_accumulation_steps"], t_cfg["max_seq_length"])
)

try:
    trainer = build_sft_trainer(effective_profile, *active_params)
    train_result = trainer.train(resume_from_checkpoint=last_ckpt)
except (torch.cuda.OutOfMemoryError, RuntimeError) as exc:
    if "out of memory" not in str(exc).lower() or effective_profile == "t4_oom_fallback":
        raise
    print(f"[OOM Safeguard Triggered] {exc}\nClearing CUDA cache and retrying with t4_oom_fallback (batch_size=1, grad_accum=16, max_seq_length=768)...")
    gc.collect()
    torch.cuda.empty_cache()
    effective_profile = "t4_oom_fallback"
    active_params = (
        fb_cfg["per_device_train_batch_size"],
        fb_cfg["per_device_eval_batch_size"],
        fb_cfg["gradient_accumulation_steps"],
        fb_cfg["max_seq_length"],
    )
    last_ckpt = get_last_checkpoint(str(DRIVE_CHECKPOINTS_DIR))
    trainer = build_sft_trainer(effective_profile, *active_params)
    train_result = trainer.train(resume_from_checkpoint=last_ckpt)

print("[Training Complete]", train_result.metrics)


## Step 5: Inspect Train/Validation Loss Logs & Save Final Adapter + Metadata to Drive
## Bước 5: Theo dõi Train/Validation Loss & Lưu Adapter Cuối + Metadata lên Drive

Persists:
1. Final LoRA adapter & tokenizer to `DRIVE_FINAL_ADAPTER_DIR` (`/content/drive/MyDrive/mimamori_companion_ai/adapters/qwen3-4b-companion-lora`)
2. `training_loss_history.json`
3. `training_metadata.json` (recording exact base model, LoRA hyperparameters, effective T4 profile, dataset review status, and final metrics)


In [ ]:
from datetime import datetime, timezone

log_history = list(trainer.state.log_history)
train_losses = [{"step": x.get("step"), "epoch": x.get("epoch"), "loss": x["loss"]} for x in log_history if "loss" in x]
eval_losses = [{"step": x.get("step"), "epoch": x.get("epoch"), "eval_loss": x["eval_loss"]} for x in log_history if "eval_loss" in x]

print("Train loss points:", train_losses)
print("Eval  loss points:", eval_losses)

# Optional loss curve plot
try:
    import matplotlib.pyplot as plt
    if train_losses or eval_losses:
        plt.figure(figsize=(7, 4))
        if train_losses:
            plt.plot([p["step"] for p in train_losses], [p["loss"] for p in train_losses], marker="o", label="train_loss")
        if eval_losses:
            plt.plot([p["step"] for p in eval_losses], [p["eval_loss"] for p in eval_losses], marker="s", label="val_loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("Mimamori Qwen3-4B QLoRA Train vs Validation Loss")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(DRIVE_EVAL_OUTPUTS_DIR / "training_loss_curve.png", dpi=150)
        plt.show()
except Exception as plot_exc:
    print(f"[Info] Skipped loss curve plot: {plot_exc}")

# Evaluate on held-out test split before releasing GPU memory
test_split_metrics = trainer.evaluate(eval_dataset=test_dataset)
print("[Test Split Metrics (test.jsonl)]", test_split_metrics)

# Qualitative sample generation on test.jsonl (first 3 records)
test_samples = []
trainer.model.eval()
prev_cache = getattr(trainer.model.config, "use_cache", False)
trainer.model.config.use_cache = True
for rec in test_records[:3]:
    prompt_msgs = (
        rec["messages"][:-1]
        if rec["messages"] and rec["messages"][-1]["role"] == "assistant"
        else list(rec["messages"])
    )
    prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        gen_out = trainer.model.generate(
            **inputs,
            max_new_tokens=int(MODEL_CFG["generation_defaults"]["eval_deterministic"]["max_new_tokens"]),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = gen_out[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    test_samples.append({
        "id": rec["id"],
        "category": rec["category"],
        "prompt": prompt_msgs[-1]["content"],
        "generated_reply": reply,
    })
    print(f"[Test Sample {rec['id']} | {rec['category']}] {reply}")
trainer.model.config.use_cache = prev_cache

# Save final LoRA adapter + tokenizer
DRIVE_FINAL_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(str(DRIVE_FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(DRIVE_FINAL_ADAPTER_DIR))

(DRIVE_FINAL_ADAPTER_DIR / "training_loss_history.json").write_text(
    json.dumps({"train_losses": train_losses, "eval_losses": eval_losses}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
(DRIVE_EVAL_OUTPUTS_DIR / "test_split_eval.json").write_text(
    json.dumps({"test_split_count": len(test_dataset), "metrics": test_split_metrics, "qualitative_samples": test_samples}, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

training_metadata = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "experiment_name": TRAIN_CFG["experiment"]["name"],
    "base_model_id": BASE_MODEL_ID,
    "adapter_dir": str(DRIVE_FINAL_ADAPTER_DIR),
    "checkpoints_dir": str(DRIVE_CHECKPOINTS_DIR),
    "hardware": GPU_SAFEGUARD,
    "effective_profile": effective_profile,
    "effective_hyperparameters": {
        "batch_size": active_params[0],
        "eval_batch_size": active_params[1],
        "gradient_accumulation_steps": active_params[2],
        "max_seq_length": active_params[3],
        "fp16": True,
        "bf16": False,
        "optim": TRAIN_CFG["training"]["optim"],
        "lora_r": LORA_CFG["r"],
        "lora_alpha": LORA_CFG["lora_alpha"],
        "lora_dropout": LORA_CFG["lora_dropout"],
        "lora_target_modules": LORA_CFG["target_modules"],
    },
    "dataset_provenance": {
        "train_count": len(train_records),
        "val_count": len(val_records),
        "test_count": len(test_records),
        "reviewed_train_count": reviewed_train,
        "unreviewed_train_count": unreviewed_train,
        "is_synthetic_smoke_test": IS_SYNTHETIC_SMOKE_TEST,
        "production_approved": bool(not IS_SYNTHETIC_SMOKE_TEST),
    },
    "library_versions": lib_versions,
    "train_metrics": train_result.metrics,
    "test_split_metrics": test_split_metrics,
}

for meta_dst in (DRIVE_FINAL_ADAPTER_DIR / "training_metadata.json", DRIVE_EVAL_OUTPUTS_DIR / "training_metadata.json"):
    meta_dst.write_text(json.dumps(training_metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"[Saved Metadata] {meta_dst}")


## Step 6: Free Training VRAM & Run Post-Training Benchmark Evaluation (`evaluate_model.py`)
## Bước 6: Giải phóng VRAM & Đánh giá Mô hình Sau Fine-Tune trên Bộ Benchmark

Unloads the training model cleanly from T4 VRAM and runs `scripts/evaluate_model.py --targets qwen3_4b_instruct_2507 --adapter-path <DRIVE_FINAL_ADAPTER_DIR> --output-prefix lora_qwen3_eval` so `05_model_comparison.ipynb` can compare Base vs LoRA-tuned Qwen3 directly.


In [ ]:
import subprocess

# Release training model & optimizer states before loading adapter in eval subprocess
del trainer, model
gc.collect()
torch.cuda.empty_cache()

eval_cmd = [
    sys.executable,
    str(AI_TRAINING_DIR / "scripts" / "evaluate_model.py"),
    "--eval-file", str(EVAL_PATH),
    "--model-config", str(MODEL_CONFIG_PATH),
    "--output-dir", str(DRIVE_EVAL_OUTPUTS_DIR),
    "--output-prefix", "lora_qwen3_eval",
    "--targets", "qwen3_4b_instruct_2507",
    "--adapter-path", str(DRIVE_FINAL_ADAPTER_DIR),
]
print("[Running Post-Training Evaluation]", " ".join(eval_cmd))
eval_proc = subprocess.run(eval_cmd, capture_output=True, text=True, check=True)
print(eval_proc.stdout.strip())
print("\n[Done] Proceed to 05_model_comparison.ipynb to inspect Base vs LoRA vs Qwen3.5 vs Groq comparisons.")
